# RLSF — register trajectory across training (val)

One rented session, one purpose: generate the **validation** split from five checkpoints of each
of the three RLSF arms — 15 files, 1,323 segments each, greedy — and get them off the box.

Nothing is scored here. COMET-Kiwi is skipped (it drags in the pinned-torch conflict, and the
register claim does not rest on it) and no judge is called (real money, no bearing on the
trajectory question). The 15 JSONLs are the deliverable; scoring runs off-box after teardown.

The base model is loaded **once** and adapters are swapped in place
(`LocalChatClient.swap_adapter`). Reloading a 7B base per checkpoint is the difference between a
two-hour job and an eight-hour one. It also puts all 15 checkpoints in one process, which is the
only place greedy decoding has reproduced byte-for-byte in this project (DEVLOG, 2026-07-31).

---
## 1 — Host, working tree, disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
import shutil
import subprocess
import sys

PY = sys.executable
ROOT = Path.cwd()
print('kernel', PY)

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

In [ ]:
import torch

# Blackwell is sm_120 and has no kernels in a cu126 wheel: the base loads and then every
# matmul raises. Checked before the adapters are fetched, not after.
cap = torch.cuda.get_device_capability(0)
cuda = tuple(int(x) for x in torch.version.cuda.split('.')[:2])
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert cuda >= (12, 8), f'torch built against CUDA {torch.version.cuda}; sm_120 needs 12.8+'
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

free_gib = shutil.disk_usage(ROOT).free / 2**30
# 15 GB base + 15 adapters at 323 MB + the outputs, with room for the pip cache.
assert free_gib > 30, f'{free_gib:.1f} GiB free is not enough for the base and 15 adapters'
print(f'{free_gib:.0f} GiB free on {ROOT}')

---
## 2 — Run parameters

In [ ]:
import json
from datetime import datetime, timedelta, timezone

import yaml

# The three arms, in the order the pre-registration reports them.
ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0', 'RLSF-Judge-High': 'w3_6.0'}

# Optimizer steps, not rollouts: num_iterations=4 passes per rollout and a save every 25
# rollouts put the ladder at 100..1200. checkpoint-1200 is the end of the run — `final` holds
# the same weights and scores identically in results/rlsf_select_*.json.
STEPS = [100, 200, 400, 800, 1200]

SPLIT = 'val'
EVAL_FILE = Path('data/splits/val.jsonl')
OUT_DIR = Path('outputs/rlsf_traj')

# 0 = the whole split. A cap here makes these files incomparable with outputs/rlsf_w3_*_val.jsonl,
# which are the fixed points the trajectory has to pass through.
SEG_LIMIT = 0

# Hours booked on this box. Section 5 stops before it rather than losing a checkpoint mid-write.
BUDGET_H = 8.0
DEADLINE = datetime.now(timezone.utc) + timedelta(hours=BUDGET_H)

PLAN = [(name, cell, step) for step in STEPS for name, cell in ARMS.items()]
print(f'{len(PLAN)} checkpoints, deadline {DEADLINE:%H:%M UTC}')

In [ ]:
# Decoding is read from an arm's own val config rather than restated here: a trajectory point is
# only comparable to the reported arm if it was generated under the same settings.
EVAL_CFG = yaml.safe_load(Path('configs/rlsf_eval_w3_2.0.yaml').read_text(encoding='utf-8'))
GEN = dict(EVAL_CFG['generator'])
GEN.pop('adapter_path')          # set per checkpoint in section 5
GEN['attn_implementation'] = 'sdpa'

assert GEN['model'] == 'Qwen/Qwen2.5-7B-Instruct', GEN['model']
assert (GEN['temperature'], GEN['top_p']) == (0.0, 1.0), 'not the locked greedy decoding'
assert (GEN['max_tokens'], GEN['seed']) == (1024, 42), GEN
assert GEN['dtype'] == 'bfloat16' and GEN['load_in_4bit'] is False, 'quantizing redefines the base'

STYLE = Path(EVAL_CFG['prompt']['style_instruction_file']).read_text(encoding='utf-8')
ROWS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
ROWS = ROWS[:SEG_LIMIT] if SEG_LIMIT else ROWS
assert len(ROWS) == 1323 or SEG_LIMIT, f'{len(ROWS)} val segments, expected 1323'
print(f"{GEN['model']}, greedy, max_tokens {GEN['max_tokens']}, sdpa")
print(f'{len(ROWS)} segments x {len(PLAN)} checkpoints = {len(ROWS) * len(PLAN):,} generations')

In [ ]:
import getpass
import logging
import os

# HF_TOKEN only. The adapter repo is private and the base model is public; nothing in this
# notebook can spend, so a rater key present here would be a mistake, not a convenience.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; this session makes no paid call'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

---
## 3 — Adapters and base weights, before the GPU is touched

Network first. Every download that happens after the base is resident is a GPU-hour spent
waiting on bandwidth.

In [ ]:
import time

from huggingface_hub import HfApi, snapshot_download

# Private model repo holding the trained arms; paths under it mirror models/ in this tree.
HF_REPO = 'prnamhr/style-aware-mt-rlsf'

# PeftModel needs these two and nothing else. The rest of a TRL checkpoint is optimizer state.
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')


def repo_dir(cell, step):
    return f'models/rlsf_grpo_{cell}/checkpoint-{step}'


PATTERNS = [f'{repo_dir(c, s)}/{f}' for _, c, s in PLAN for f in ADAPTER_FILES]

# Listed before anything is fetched: a missing checkpoint found 40 minutes into the download
# is a missing checkpoint found after the meter started.
present = set(HfApi().list_repo_files(HF_REPO, token=os.environ['HF_TOKEN']))
missing = [p for p in PATTERNS if p not in present]
assert not missing, f'{len(missing)} files absent from {HF_REPO}: {missing[:4]}'
print(f'all {len(PATTERNS)} adapter files present in {HF_REPO}')

In [ ]:
t0 = time.perf_counter()
snapshot_download(HF_REPO, local_dir='.', allow_patterns=PATTERNS,
                  token=os.environ['HF_TOKEN'], max_workers=8)
adapters_s = time.perf_counter() - t0

ADAPTERS = {(cell, step): Path(repo_dir(cell, step)) for _, cell, step in PLAN}
for path in ADAPTERS.values():
    assert (path / 'adapter_config.json').exists(), path
print(f'{len(ADAPTERS)} adapters in {adapters_s / 60:.1f} min')

In [ ]:
# The base too, so section 4 loads from disk. ~15 GB in bf16 safetensors.
t0 = time.perf_counter()
snapshot_download(GEN['model'], allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'],
                  token=os.environ['HF_TOKEN'], max_workers=8)
print(f"{GEN['model']} cached in {(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
import hashlib

MANIFEST = {'repo': HF_REPO, 'base': GEN['model'], 'generator': GEN, 'checkpoints': {}}

for (cell, step), path in ADAPTERS.items():
    conf = json.loads((path / 'adapter_config.json').read_text(encoding='utf-8'))
    assert conf['r'] == 32 and conf['lora_alpha'] == 64, (cell, step, conf['r'])
    assert conf['base_model_name_or_path'].endswith(GEN['model'].split('/')[-1]), conf
    digest = hashlib.sha256((path / 'adapter_model.safetensors').read_bytes()).hexdigest()
    MANIFEST['checkpoints'][f'{cell}_step{step}'] = {'adapter': str(path), 'sha256': digest}

# 15 distinct sets of weights, or an upload put the same checkpoint under two names and the
# trajectory would show a flat stretch that is really a copy.
digests = [v['sha256'] for v in MANIFEST['checkpoints'].values()]
assert len(set(digests)) == len(digests), 'two checkpoints have identical weights'
for tag, v in MANIFEST['checkpoints'].items():
    print(f"{tag:16s} {v['sha256'][:12]}  {v['adapter']}")

---
## 4 — The gate

The base load is the one in section 5, not a throwaway: the probe runs on the client the
trajectory pass then keeps using.

Reference point: the selection session in `notebooks/rlsf_eval_gpu.ipynb` measured **2.24 s per
val segment** on a 4090 at these settings. Mean completion is 33 tokens, so about half of that is
per-call overhead rather than decode, and only the decode half scales with bandwidth. Fifteen
checkpoints at 2.24 s is 12.3 h; this box has to reach roughly 1.5 s/segment for the full ladder
to fit an 8-hour booking.

In [ ]:
from src.infer.run import build_zeroshot_user, make_client

FIRST = (ARMS['RL-Metric'], STEPS[0])

t0 = time.perf_counter()
client = make_client({**GEN, 'adapter_path': str(ADAPTERS[FIRST])})
load_s = time.perf_counter() - t0

PROBE_N = 8
t0 = time.perf_counter()
for row in ROWS[:PROBE_N]:
    client.complete(STYLE, build_zeroshot_user(row['input']))
seg_s = (time.perf_counter() - t0) / PROBE_N

t0 = time.perf_counter()
client.swap_adapter(str(ADAPTERS[(ARMS['RLSF-Judge'], STEPS[0])]))
client.swap_adapter(str(ADAPTERS[FIRST]))
swap_s = (time.perf_counter() - t0) / 2

print(f'{load_s:.0f}s base load, {seg_s:.2f}s per segment, {swap_s:.1f}s per adapter swap')
print(f'{torch.cuda.max_memory_reserved() / 2**30:.1f} GiB reserved')

In [ ]:
ckpt_h = (len(ROWS) * seg_s + swap_s) / 3600
total_h = ckpt_h * len(PLAN)
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{ckpt_h:.2f} h per checkpoint, {total_h:.1f} h for {len(PLAN)}, '
      f'{left_h:.1f} h left of the booking')

if total_h > 0.9 * left_h:
    print('\nDoes not fit. Levers, in the order they cost the least:')
    print('  - drop step 800 from STEPS: the ladder keeps 100/200/400/1200 and its endpoints')
    print('  - drop the w3_6.0 arm: the omega contrast survives on 0.0 vs 2.0')
    print('  - book more hours; SEG_LIMIT is not a lever, it breaks comparability with the arms')
else:
    print('\nFits. Section 5 may start.')

---
## 5 — The trajectory pass

Step-major order: all three arms at step 100, then all three at 200. A session that dies early
then leaves complete step-slices, which are readable as a trajectory; arm-major would leave one
finished arm and two absent ones.

Every file resumes. Rows are appended and fsync'd one at a time, so an interrupted checkpoint
loses at most the in-flight segment.

In [ ]:
from src.eval._io import read_completed_jsonl

OUT_DIR.mkdir(parents=True, exist_ok=True)


def out_path(cell, step):
    return OUT_DIR / f'rlsf_{cell}_step{step}_{SPLIT}.jsonl'


def generate(client, cell, step):
    """Generate the split from the adapter the client currently holds, resuming a partial file."""
    path = out_path(cell, step)
    assert client.adapter_path == str(ADAPTERS[(cell, step)]), (client.adapter_path, cell, step)

    done = read_completed_jsonl(path)
    assert len(done) <= len(ROWS), f'{path}: {len(done)} rows, {len(ROWS)} segments'
    for j, rec in enumerate(done):
        assert rec['input'] == ROWS[j]['input'], f'resume misalignment in {path} at segment {j}'
    if len(done) == len(ROWS):
        return 0

    t0, failures = time.perf_counter(), 0
    with path.open('a', encoding='utf-8') as f:
        for i in range(len(done), len(ROWS)):
            row = ROWS[i]
            try:
                prediction, error = client.complete(STYLE, build_zeroshot_user(row['input'])), None
            except Exception as e:  # one bad segment must not discard the checkpoint
                prediction, error = '', f'{type(e).__name__}: {e}'
                failures += 1
            record = {
                'input': row['input'],
                'output': row['output'],
                'prediction': prediction,
                # The arms' val files carry 'peft': RLSF adapts the policy, not the prompt, so
                # the prompt condition is the same one. The trajectory point is in arm/step.
                'condition': 'peft',
                'model': GEN['model'],
                'metadata': row.get('metadata', {}),
                'arm': cell,
                'step': step,
                'adapter': str(ADAPTERS[(cell, step)]),
            }
            if error:
                record['error'] = error
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            f.flush()
            os.fsync(f.fileno())
            if (i + 1) % 200 == 0:
                rate = (time.perf_counter() - t0) / (i + 1 - len(done))
                print(f'  {i + 1}/{len(ROWS)}  {rate:.2f}s/seg')
    if failures:
        print(f'  WARNING: {failures} segments recorded an error and an empty prediction')
    return time.perf_counter() - t0

In [ ]:
MANIFEST_PATH = OUT_DIR / 'manifest.json'
skipped = []

# A resumed session rebuilds MANIFEST from the weights on disk; the timings of checkpoints
# finished before the interruption survive only if they are carried forward here.
if MANIFEST_PATH.exists():
    prior = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))['checkpoints']
    for tag, entry in MANIFEST['checkpoints'].items():
        entry.update({k: v for k, v in prior.get(tag, {}).items() if k not in entry})

for name, cell, step in PLAN:
    tag = f'{cell}_step{step}'
    left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
    complete = len(read_completed_jsonl(out_path(cell, step))) == len(ROWS)
    if not complete and left_h < ckpt_h:
        skipped.append(tag)
        print(f'{tag}: {left_h:.1f} h left, {ckpt_h:.2f} h needed — not started')
        continue

    if client.adapter_path != str(ADAPTERS[(cell, step)]):
        client.swap_adapter(str(ADAPTERS[(cell, step)]))
    print(f'{tag} ({name})  {left_h:.1f} h left')
    elapsed = generate(client, cell, step)

    entry = MANIFEST['checkpoints'][tag]
    entry.update(arm=name, output=str(out_path(cell, step)))
    if elapsed:  # 0 means the file was already complete when this session reached it
        entry['seconds'] = round(elapsed, 1)
        entry['finished'] = datetime.now(timezone.utc).isoformat(timespec='seconds')
    # Written after every checkpoint: a box that disappears still leaves a record of
    # which weights wrote which file.
    MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2), encoding='utf-8')
    print(f'  done in {elapsed / 60:.0f} min' if elapsed else '  already complete, skipped')

print(f'\n{len(PLAN) - len(skipped)}/{len(PLAN)} checkpoints generated')
if skipped:
    print('not generated:', ', '.join(skipped))

---
## 6 — Verify before teardown

The last chance to catch a truncated or misaligned file while the weights that wrote it are
still on the box.

In [ ]:
FILES = {}
for _, cell, step in PLAN:
    path = out_path(cell, step)
    if not path.exists():
        continue
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(ROWS), f'{path}: {len(rows)} rows, expected {len(ROWS)}'
    assert all(a['input'] == b['input'] for a, b in zip(rows, ROWS)), f'{path}: source misalignment'
    assert not [r for r in rows if r.get('error')], f'{path}: a segment recorded an error'
    assert not [r for r in rows if not r['prediction'].strip()], f'{path}: empty prediction'
    assert {(r['arm'], r['step']) for r in rows} == {(cell, step)}, f'{path}: mixed provenance'
    FILES[(cell, step)] = (path, rows)
print(f'{len(FILES)} files, {len(ROWS)} aligned rows each, no errors, no empties')

In [ ]:
from sacrebleu.metrics import CHRF

from src.eval.quick import _marker_rate

# Integrity, not a result: chrF says the checkpoint still translates, marker_rate says the
# register moved. The claim is scored off-box against the held-out feature split.
chrf = CHRF()
print(f"{'arm':10s} {'step':>5s} {'chrF':>7s} {'marker_rate':>12s}")
for (cell, step), (_, rows) in sorted(FILES.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    preds = [r['prediction'] for r in rows]
    score = chrf.corpus_score(preds, [[r['output'] for r in rows]]).score
    print(f'{cell:10s} {step:5d} {score:7.2f} {_marker_rate(preds):12.2f}')

In [ ]:
# The seal: nothing here read the test split, and nothing here could spend.
for (path, _) in FILES.values():
    assert 'test' not in path.name, path
usage = client.usage.summary()
assert usage['cost_usd'] == 0.0, usage
print(f"{usage['calls']:,} local calls, {usage['completion_tokens']:,} tokens generated, "
      f"$0.00 spent")

---
## 7 — Teardown

Pull the archive down, confirm it opens locally, *then* destroy the instance. Scoring these 15
files needs `src/eval/heldout_decomp.py` extended — its `OMEGA` map is keyed by condition and has
no entry for a trajectory tag — which is off-box work and no reason to keep a GPU running.

In [ ]:
ARCHIVE = Path(f'rlsf_traj_{SPLIT}.tar.gz')
subprocess.run(['tar', 'czf', str(ARCHIVE), '-C', str(OUT_DIR.parent), OUT_DIR.name], check=True)
print(f'{ARCHIVE}  {ARCHIVE.stat().st_size / 2**20:.1f} MiB')
print(f'{len(FILES)} outputs + manifest.json')